# The Acceleration Problem for a Fuchs-Class Warp Shell (Task 2A.10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bshepp/alcubierre/blob/main/acceleration.ipynb)

**Runtime:** local (default, cells 1–8, 13–16) · Colab CPU · HF Jobs `cpu-upgrade` (cells 9–12 require `requirements-gw.txt`).

Package 3 of the Path 2A plan. Depends on `israel_junction.ipynb` (the $\lambda_*$ acceleration obstruction identified in Part B, cell 13) and `thickness_bound.ipynb` (the $\kappa \beta/C$ thickness bound).

**Goal.** Determine whether a Fuchs-class warp shell can be accelerated — i.e. whether $v_{\rm warp}$ can be changed in time, or the shell's centre-of-mass can be translated — without reintroducing negative-energy regions during the transient. The analysis is organised around ADM 4-momentum conservation plus a catalogue of candidate mechanisms, and the one mechanism amenable to real numerics today (gravitational-wave recoil) is backed by a quantitative estimate for Fuchs-class parameters.

**Structure.**

- **Cells 1–4 (ADM framework).** Compute the ADM 4-momentum of a Fuchs shell symbolically; derive the conservation-law obstruction to spontaneous acceleration.
- **Cells 5–8 (candidate mechanisms).** Analyse three plausible acceleration mechanisms (shift spin-up, mass ejection / photon-rocket, gravitational-wave recoil) for DEC compatibility and ADM-momentum consistency.
- **Cells 9–12 (GW-recoil numerics, HF Jobs scope).** Quantitative GW-recoil estimate with two parallel approaches (SXS catalog waveform rescaling and a post-Newtonian shell-mass analog); deliverable is a numerical ceiling on $\Delta v_{\max}$ for Fuchs parameters.
- **Cells 13–16 (synthesis).** Obstruction-catalog table; Schuster-Santiago-Visser comparison; Varma et al. 2022 quantitative comparison using the cell-9/10 output; conclusion on which mechanism (if any) survives.

**Key deliverable.** An obstruction catalog table plus a numerical ceiling on GW-recoil $\Delta v_{\max}$ for Fuchs-class shells, concretising (or closing) outcome scenario (C) of `QUANTUM_CLASSICAL_BRIDGE.md` §6.


In [ ]:
import os, sys, subprocess

if "google.colab" in sys.modules or os.environ.get("HF_JOB"):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
else:
    print("Local runtime detected; skipping pip install.")

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols, Function, diff, simplify, series, sqrt, pi, Rational, integrate, cos, sin

print(f"SymPy {sp.__version__}, NumPy {np.__version__}")


Local runtime detected; skipping pip install.


SymPy 1.14.0, NumPy 2.4.2


## Part 1 — ADM 4-Momentum Framework (cells 2–5)

The ADM decomposition provides a conserved 4-momentum $P_\mu^{\rm ADM}$ at spatial infinity for any asymptotically flat spacetime. For a Fuchs-class shell embedded in asymptotically flat vacuum, $P_\mu^{\rm ADM} = (-E_{\rm tot}, \vec P_{\rm tot})$ where

$$E_{\rm tot} = \frac{1}{16\pi G} \oint_{\infty} (\partial_j h_{ij} - \partial_i h_{jj})\,dS^i,$$

$$P^i_{\rm tot} = \frac{1}{8\pi G} \oint_{\infty} (K_{ij} - K\,h_{ij})\,dS^j,$$

with $h_{ij}$ the spatial 3-metric and $K_{ij}$ its extrinsic curvature in the ADM slicing. For an initially-static configuration ($\beta^i = 0$ on the slice), $P^i_{\rm tot} = 0$ by construction.

**Obstruction from conservation.** If the shell starts at rest ($P^i_{\rm tot}(t_0) = 0$) and the exterior remains vacuum at infinity at all later times, then $P^i_{\rm tot}(t) = 0$ for all $t$. Any non-zero coordinate velocity $v_{\rm warp}(t) \neq 0$ of the shell's centre must therefore be balanced by an equal-and-opposite momentum elsewhere: either in matter fluxes leaving the shell, in the stress-energy of a non-vacuum exterior, or (the only vacuum-compatible option) in outgoing gravitational radiation.

This is the **acceleration obstruction** in its covariant, gauge-independent form. It rules out "self-accelerating" warp shells in asymptotically flat vacuum. Whether any of the three permitted mechanisms (shift spin-up with non-vacuum exterior, mass ejection, GW emission) is compatible with DEC is the question addressed in cells 5–8.


In [ ]:
# Cell 3: Symbolic ADM energy of a Fuchs shell (weak-field / isotropic-gauge check).
# For a Schwarzschild exterior matched to a regular interior across r=R, the ADM
# energy equals the Schwarzschild mass M. We verify this by computing the ADM
# surface integral in the exterior and taking r -> infinity.

r, M_sym, G_sym = symbols('r M G', positive=True)

# Isotropic Schwarzschild exterior: ds^2 = -A(r) dt^2 + B(r) delta_ij dx^i dx^j
# with A = ((1 - GM/(2r))/(1 + GM/(2r)))^2, B = (1 + GM/(2r))^4  (G = c = 1).
B = (1 + G_sym * M_sym / (2 * r))**4
# h_ij = B(r) delta_ij.  In isotropic coords the ADM integrand simplifies to
#   (partial_j h_ij - partial_i h_jj) n^i  =  -2 dB/dr  (pointed radially outward),
# and the area element is 4 pi r^2 * (B^(1/2))^2 = 4 pi r^2 B (but to leading
# order at infinity only the monopole matters and B -> 1, so we use
#   E_ADM = (1/(16 pi G)) * lim_{r->inf} [ -4 pi r^2 * 2 dB/dr ].

dBdr = diff(B, r)
integrand = -2 * dBdr * (4 * pi * r**2)
E_ADM = integrand / (16 * pi * G_sym)
E_ADM_at_inf = sp.limit(E_ADM, r, sp.oo)

print("ADM energy of Schwarzschild exterior (isotropic gauge):")
print("  E_ADM(r) =", simplify(E_ADM))
print("  E_ADM(r -> inf) =", E_ADM_at_inf)
assert simplify(E_ADM_at_inf - M_sym) == 0, "ADM energy failed to reduce to M"
print("  => E_ADM = M  (textbook result; framework validated).")


ADM energy of Schwarzschild exterior (isotropic gauge):
  E_ADM(r) = M*(G*M + 2*r)**3/(8*r**3)
  E_ADM(r -> inf) = M
  => E_ADM = M  (textbook result; framework validated).


In [ ]:
# Cell 4: ADM linear momentum and the obstruction formalised.
#
# For a t = const slice of the form
#   ds^2 = -(N^2 - N_i N^i) dt^2 + 2 N_i dt dx^i + h_ij dx^i dx^j
# the ADM linear momentum is
#   P^i = (1/(8 pi G)) * integral_{S_inf} pi^{ij} dS_j,
# with pi^{ij} = -sqrt(h) (K^{ij} - K h^{ij}).
#
# (A) Initially-static Fuchs slice: by construction K_{ij} = 0 (the shell is in
#     equilibrium, no shift, no time-dependence), so P^i = 0 identically.
#
# (B) After some transient: *if* the exterior is asymptotically flat and vacuum,
#     then dP^i/dt = (boundary flux of matter stress-energy) + (GW flux).
#     Vacuum + no matter efflux  =>  P^i conserved  =>  P^i(t) = 0 forever.
#
# This is the key obstruction theorem for the rest of the notebook.

print("ADM linear momentum obstruction theorem (stated symbolically):")
print()
print("  Assumptions:")
print("    (i)   Initial slice Sigma_0 has K_{ij} = 0 (shell at rest, no shift).")
print("    (ii)  Spacetime is asymptotically flat with vacuum exterior for all t.")
print("    (iii) No matter flux escapes to infinity.")
print()
print("  Conclusion:")
print("    P^i_ADM(t) = 0 for all t > t_0.")
print()
print("  Corollary:  Any nonzero centre-of-mass velocity v^i(t) must be balanced")
print("  by either (a) matter flux (drops assumption iii), (b) non-vacuum exterior")
print("  (drops ii), or (c) gravitational-wave momentum flux dP^i/dt|_GW (which is")
print("  generically nonzero for quadrupole-asymmetric sources).")
print()
print("  These three escape hatches are analysed individually in cells 5-8.")


ADM linear momentum obstruction theorem (stated symbolically):

  Assumptions:
    (i)   Initial slice Sigma_0 has K_{ij} = 0 (shell at rest, no shift).
    (ii)  Spacetime is asymptotically flat with vacuum exterior for all t.
    (iii) No matter flux escapes to infinity.

  Conclusion:
    P^i_ADM(t) = 0 for all t > t_0.

  Corollary:  Any nonzero centre-of-mass velocity v^i(t) must be balanced
  by either (a) matter flux (drops assumption iii), (b) non-vacuum exterior
  (drops ii), or (c) gravitational-wave momentum flux dP^i/dt|_GW (which is
  generically nonzero for quadrupole-asymmetric sources).

  These three escape hatches are analysed individually in cells 5-8.


In [ ]:
# Cell 5: Numerical ADM mass for Fuchs-class parameters (calibrates later scales).
# Fuchs et al. 2024: R ~ 15 m, M ~ 4.49e27 kg, beta ~ 0.02, Delta ~ 10 m.
# We compute the ADM mass in SI (should equal M) and the corresponding
# Schwarzschild radius, Schwarzschild compactness, and ADM rest-energy.

G_SI = 6.67430e-11
c_SI = 2.99792458e8

M_F     = 4.49e27      # kg
R_F     = 15.0          # m
beta_F  = 0.02
Delta_F = 10.0          # m

r_s_F = 2 * G_SI * M_F / c_SI**2
C_F   = r_s_F / R_F
E_ADM_F = M_F * c_SI**2

print(f"Fuchs-class shell numerical invariants:")
print(f"  M         = {M_F:.3e} kg")
print(f"  R         = {R_F:.3e} m")
print(f"  beta      = {beta_F}")
print(f"  Delta     = {Delta_F} m")
print(f"  r_s       = 2GM/c^2  = {r_s_F:.3e} m")
print(f"  C = r_s/R            = {C_F:.3e}")
print(f"  E_ADM = Mc^2         = {E_ADM_F:.3e} J")
print()
print("These invariants are the *budget* from which any ADM-permitted acceleration")
print("mechanism must draw.  In particular, any recoil delta-v satisfies")
print()
print("  |delta-v| <= delta-P_radiated / M,")
print()
print("which is the quantity we estimate numerically in cells 9-12.")


Fuchs-class shell numerical invariants:
  M         = 4.490e+27 kg
  R         = 1.500e+01 m
  beta      = 0.02
  Delta     = 10.0 m
  r_s       = 2GM/c^2  = 6.669e+00 m
  C = r_s/R            = 4.446e-01
  E_ADM = Mc^2         = 4.035e+44 J

These invariants are the *budget* from which any ADM-permitted acceleration
mechanism must draw.  In particular, any recoil delta-v satisfies

  |delta-v| <= delta-P_radiated / M,

which is the quantity we estimate numerically in cells 9-12.


## Part 2 — Three Candidate Acceleration Mechanisms (cells 6–9)

The ADM conservation corollary permits exactly three ways the shell can acquire momentum:

1. **Mechanism A — shift spin-up with non-vacuum exterior.** The warp bubble's shift vector $\beta^i$ is quasi-statically turned on. Momentum is deposited in the exterior matter/field that breaks the "vacuum" assumption.
2. **Mechanism B — mass ejection (photon rocket).** The shell throws matter or radiation backwards; by conservation the shell's centre gains forward momentum. This is simply Tsiolkovsky physics dressed up in GR, and the DEC compatibility of the *ejecta* is the critical question.
3. **Mechanism C — gravitational-wave recoil.** A time-asymmetric quadrupole pattern radiates net linear momentum to infinity; by conservation the shell recoils. This is the only mechanism compatible with *both* a vacuum exterior and the DEC, but the magnitude is generically tiny for non-merger sources.

We analyse each in turn and map it back to Part B's $\lambda_*$ obstruction.


In [5]:
# Cell 7: Mechanism A -- shift spin-up with non-vacuum exterior.
#
# In Part B (israel_junction.ipynb, cell 28) we found that if v_ext and v_int are
# allowed to differ, the DEC fails for lambda = v_ext / v_int < lambda* ~ 0.55.
# Physically: during a transient in which v_int is ramped from 0 to v_final
# while v_ext lags, the shell passes through a DEC-violating regime unless the
# ratio stays above lambda*.
#
# Mechanism A asks: can we avoid this by filling the exterior with real matter
# that carries momentum, so that v_ext and v_int can be ramped in lockstep?

print("Mechanism A -- shift spin-up, non-vacuum exterior")
print("-" * 60)
print()
print("Conservation constraint:  P_total = P_shell + P_exterior_matter = 0.")
print()
print("If the shell's internal v_int is ramped from 0 to v_f, then")
print("   P_shell ~= M_shell * gamma(v_f) * v_f.")
print("For Fuchs parameters (M = 4.49e27 kg, v_f = beta*c = 6e6 m/s),")

M_F_val     = 4.49e27
beta_F_val  = 0.02
c_SI_val    = 2.99792458e8
v_f_val     = beta_F_val * c_SI_val
gamma_f     = 1.0 / np.sqrt(1 - beta_F_val**2)
P_shell_Fuchs = M_F_val * gamma_f * v_f_val
print(f"   P_shell = {P_shell_Fuchs:.3e} kg m/s")
print()
print("The exterior must carry -P_shell.  If the exterior medium has density")
print("rho_ext over a volume V_ext at mean velocity v_ext_m, then")
print("   |rho_ext * V_ext * v_ext_m| = P_shell.")
print()
print("DEC on the exterior matter requires rho_ext > 0 and |rho_ext * v_ext_m|")
print("consistent with ordinary matter.  Mechanism A is therefore DEC-compatible")
print("IF AND ONLY IF:")
print("  (a) the exterior is pre-loaded with positive-energy matter totalling a")
print("      substantial fraction of the shell mass, AND")
print("  (b) that matter is spooled up / spooled down coherently with the interior.")
print()
print("Assessment:  technically DEC-compatible, but this is not 'warp drive' --")
print("it is 'push-from-a-wall'.  The shell cannot accelerate in genuinely empty")
print("space under Mechanism A.  It is indistinguishable from a rocket whose")
print("exhaust happens to be frozen into the surrounding medium.")


Mechanism A -- shift spin-up, non-vacuum exterior
------------------------------------------------------------

Conservation constraint:  P_total = P_shell + P_exterior_matter = 0.

If the shell's internal v_int is ramped from 0 to v_f, then
   P_shell ~= M_shell * gamma(v_f) * v_f.
For Fuchs parameters (M = 4.49e27 kg, v_f = beta*c = 6e6 m/s),
   P_shell = 2.693e+34 kg m/s

The exterior must carry -P_shell.  If the exterior medium has density
rho_ext over a volume V_ext at mean velocity v_ext_m, then
   |rho_ext * V_ext * v_ext_m| = P_shell.

DEC on the exterior matter requires rho_ext > 0 and |rho_ext * v_ext_m|
consistent with ordinary matter.  Mechanism A is therefore DEC-compatible
IF AND ONLY IF:
  (a) the exterior is pre-loaded with positive-energy matter totalling a
      substantial fraction of the shell mass, AND
  (b) that matter is spooled up / spooled down coherently with the interior.

Assessment:  technically DEC-compatible, but this is not 'warp drive' --
it is 'push-fr

In [6]:
# Cell 8: Mechanism B -- mass ejection / photon rocket.
# Tsiolkovsky rocket equation:  delta-v / v_ex = ln(m_0 / m_f).
# For a photon rocket v_ex = c.  For a matter rocket v_ex < c.  Either way the
# DEC on the ejecta is trivial (ordinary matter/radiation) but the problem is
# the MASS BUDGET.

print("Mechanism B -- mass ejection / photon rocket")
print("-" * 60)
print()
# Photon-rocket ceiling: to reach v_f = beta_F * c requires mass ratio
# exp(beta_F), which is trivially achievable.  But to reach relativistic v,
# we need ln(m0/mf) = artanh(beta).  For beta -> 1 the required ratio diverges.
betas = np.array([0.01, 0.1, 0.5, 0.9, 0.99, 0.999])
ratios = np.exp(np.arctanh(betas))   # m_0 / m_f for photon rocket
print("Photon-rocket mass ratio to reach final velocity v_f = beta * c:")
print(f"  {'beta':>8}  {'m_0/m_f':>12}")
for b, rr in zip(betas, ratios):
    print(f"  {b:8.3f}  {rr:12.3e}")
print()
print("For Fuchs-class beta = 0.02, the photon-rocket mass ratio is a trivial")
print(f"  m_0/m_f = {float(np.exp(np.arctanh(0.02))):.4f}")
print("so Mechanism B can in principle accelerate a Fuchs shell to its own warp")
print("speed with essentially zero mass loss.")
print()
print("But: what is being accelerated is now the *combined* system (shell + fuel)")
print("which initially has mass much larger than the warp-shell-alone mass.  The")
print("DEC analysis of israel_junction.ipynb was for a *shell* of mass M; if the")
print("fuel is carried inside the shell, the effective C = 2GM_tot/(R c^2) is")
print("larger, and the thickness_bound.ipynb scaling law tells us the required")
print("Delta_min / R scales as beta / C -- DECREASES with larger M_tot.  So")
print("carrying fuel does NOT break Path 2A feasibility; it just moves the design")
print("point.")
print()
print("Assessment:  Mechanism B is DEC-compatible, ADM-consistent, and")
print("thickness-bound-compatible.  It is however an ordinary rocket, not a warp")
print("drive in any meaningful sense.  A 'warp shell accelerated by a chemical")
print("rocket stage' is technically a Fuchs-class warp drive in accelerating")
print("motion, and this is perhaps the most honest reading of the Fuchs et al.")
print("2024 result: they built a geometry that is static-viable, and the only")
print("way to translate it is to bolt a rocket to it.")


Mechanism B -- mass ejection / photon rocket
------------------------------------------------------------

Photon-rocket mass ratio to reach final velocity v_f = beta * c:
      beta       m_0/m_f
     0.010     1.010e+00
     0.100     1.106e+00
     0.500     1.732e+00
     0.900     4.359e+00
     0.990     1.411e+01
     0.999     4.471e+01

For Fuchs-class beta = 0.02, the photon-rocket mass ratio is a trivial
  m_0/m_f = 1.0202
so Mechanism B can in principle accelerate a Fuchs shell to its own warp
speed with essentially zero mass loss.

But: what is being accelerated is now the *combined* system (shell + fuel)
which initially has mass much larger than the warp-shell-alone mass.  The
DEC analysis of israel_junction.ipynb was for a *shell* of mass M; if the
fuel is carried inside the shell, the effective C = 2GM_tot/(R c^2) is
larger, and the thickness_bound.ipynb scaling law tells us the required
Delta_min / R scales as beta / C -- DECREASES with larger M_tot.  So
carrying fuel 

In [7]:
# Cell 9: Mechanism C -- gravitational-wave recoil.
# The only mechanism that works in a *genuinely empty* exterior.  GW momentum
# flux (Thorne 1980 multipole formula):
#
#   dP^i / dt = (2/5) (G/c^7) <(d^3 I_{ij}/dt^3)(d^3 I_{ik}/dt^3)> * (1/c^7)  [mass quad]
#                + higher multipoles
#
# For a single quasi-spherical pulsating shell, the mass quadrupole oscillation
# is set by the asymmetry induced by the shift beta^r ~ cos(theta).  We here
# derive an order-of-magnitude ceiling; the NR-backed quantitative estimate
# lives in cells 10-13.

print("Mechanism C -- gravitational-wave recoil")
print("-" * 60)
print()
print("Characteristic scale.  Dimensional analysis:")
print("   dP/dt  ~  (G/c^7) * M^2 R^4 omega^6")
print("where omega is the characteristic frequency of the asymmetric quadrupole.")
print()
print("For a Fuchs shell 'turned on' over timescale tau, omega ~ 1/tau, so over")
print("the full event:")
print("   delta-P  ~  (G/c^7) * M^2 R^4 / tau^5.")
print()
print("For Fuchs parameters and various tau:")
G_SI_val = 6.67430e-11
c_SI_val = 2.99792458e8
M_F_val2 = 4.49e27
R_F_val  = 15.0
print(f"  {'tau (s)':>10}  {'delta-P (kg m/s)':>20}  {'delta-v = delta-P/M (m/s)':>28}")
for tau in [1e-3, 1.0, 1e3, 1e6]:
    dP = (G_SI_val / c_SI_val**7) * M_F_val2**2 * R_F_val**4 / tau**5
    dv = dP / M_F_val2
    print(f"  {tau:10.1e}  {dP:20.3e}  {dv:28.3e}")
print()
print("Even for tau = 1 ms (a violent 'snap-on' at solar-mass time scales), the")
print("delta-v deposited by GW recoil is utterly negligible compared to any")
print("warp-relevant velocity (~ beta c = 6e6 m/s).  Mechanism C is DEC- and")
print("ADM-compatible but phenomenologically useless for Fuchs parameters.")
print()
print("The honest quantitative question, which we address with real NR data in")
print("the next section, is:  what is the *maximum* recoil delta-v achievable by")
print("any Fuchs-class shell under any reasonable dynamic history? For merging")
print("black holes the Varma et al. 2022 record is ~5000 km/s; rescaling to")
print("Fuchs-class compactness and quadrupole asymmetry will give a numerical")
print("ceiling.")


Mechanism C -- gravitational-wave recoil
------------------------------------------------------------

Characteristic scale.  Dimensional analysis:
   dP/dt  ~  (G/c^7) * M^2 R^4 omega^6
where omega is the characteristic frequency of the asymmetric quadrupole.

For a Fuchs shell 'turned on' over timescale tau, omega ~ 1/tau, so over
the full event:
   delta-P  ~  (G/c^7) * M^2 R^4 / tau^5.

For Fuchs parameters and various tau:
     tau (s)      delta-P (kg m/s)     delta-v = delta-P/M (m/s)
     1.0e-03             3.130e+05                     6.971e-23
     1.0e+00             3.130e-10                     6.971e-38
     1.0e+03             3.130e-25                     6.971e-53
     1.0e+06             3.130e-40                     6.971e-68

Even for tau = 1 ms (a violent 'snap-on' at solar-mass time scales), the
delta-v deposited by GW recoil is utterly negligible compared to any
warp-relevant velocity (~ beta c = 6e6 m/s).  Mechanism C is DEC- and
ADM-compatible but phenomenolo

## Part 3 — GW-Recoil Quantitative Estimate (cells 10–13, HF Jobs scope)

Only Mechanism C (GW recoil) is fully vacuum- and DEC-compatible, so it is the only mechanism where a hard numerical ceiling on $\Delta v_{\max}$ can be derived from first principles. Cell 9 already gave a dimensional ceiling; here we harden it with two independent approaches:

- **Approach A — SXS-catalog rescaling.** The Simulating Extreme Spacetimes collaboration has published thousands of NR waveforms for binary black hole (BBH) mergers. The maximum recoil measured to date is ~5000 km/s (Varma et al. 2022, sxs:BBH_SKS_d14.3_q1.22_sA_0.51_-0.62_0.16_sB_-0.51_0.54_-0.69). Rescaling this to Fuchs-shell compactness and mass gives a ceiling with a well-understood systematic: the recoil kick $v_{\rm kick}$ scales roughly as $f_{\rm asym}^2$ where $f_{\rm asym}$ is the dimensionless quadrupole asymmetry. BBH mergers saturate at $f_{\rm asym} \sim 1$ during merger; a Fuchs shell has $f_{\rm asym} \sim \beta$ in the dipole projection (cell 11 of israel_junction.ipynb). The rescaling is therefore roughly $v_{\rm kick}^{\rm Fuchs} \sim v_{\rm kick}^{\rm BBH} \cdot (\beta_{\rm Fuchs})^2$.
- **Approach B — post-Newtonian quadrupole analog.** Treat the Fuchs shell as a non-spinning quadrupole with time-dependent $I_{ij}(t)$ driven by a realistic acceleration profile $v_{\rm warp}(t) = v_f [1 - \exp(-t/\tau)]$, and integrate the PN momentum-flux formula. This gives an order-of-magnitude cross-check independent of the SXS rescaling.

**HF Jobs scope.** The SXS waveform download and processing is I/O-heavy and the PN integration is CPU-heavy but easily parallelised over $\tau$ and $\beta$. Both fit within a `cpu-upgrade` HF Job with `requirements-gw.txt`. Locally we run a coarse preview (cell 11); a dedicated sweep module `hf_jobs/sweeps/gw_recoil.py` scales this to HF Jobs (cell 13 below describes how to invoke it).


In [ ]:
# Cell 11: Approach B -- post-Newtonian quadrupole analog (local, no GW stack).
#
# A Fuchs shell ramped from rest to v_f = beta * c over time tau has a
# time-dependent mass dipole (in the rest frame of the lab).  The mass dipole
# cannot radiate (it is a total-derivative of P_shell, which is eventually
# constant), but the time derivative of the mass *quadrupole* arising from the
# shell's finite size CAN radiate.
#
# For a uniform shell of radius R executing a coordinate displacement z(t) along
# the z-axis, the mass quadrupole is
#    I_zz(t) - I_xx(t)  =  (2/3) M R^2   (shell's own, time-independent)
#                        +  M [z(t)^2 - <x^2>(t)]  (translation)
# The constant piece doesn't radiate.  The translational piece gives
#    I_zz(t) - I_xx(t) -> M [z(t)^2 - 0] = M z(t)^2 (isotropic shell assumption).
#
# The Thorne momentum-flux formula (Newtonian order) for a quadrupole along z is
#    dP^z/dt = (2/5) (G/c^7) I_zz^{(3)} [ ... ] = 0 identically for axially
# symmetric sources (the quadrupole and current-quadrupole have to be beat
# against each other to give a momentum flux).
#
# So the analog we actually need is a *binary* of two Fuchs shells, or a Fuchs
# shell with a tiny asymmetric companion -- in either case the PN momentum-flux
# formula for a binary gives (Fitchett 1983, Blanchet 2014):
#
#    F_P = - (8 G^4 / (105 c^7)) * (M1 M2)^2 (M1 - M2) / a^5 * n_hat
#          at lowest order
#
# We will use this formula with M2 << M1 (a small "beacon" mass) to bound the
# recoil.

G_SI_val = 6.67430e-11
c_SI_val = 2.99792458e8

def dP_dt_binary_fitchett(M1, M2, a):
    """Newtonian-order linear momentum radiated by an unequal-mass circular binary.
    Blanchet 2014 Eq. 7.38 (leading PN term); magnitude, in SI."""
    if M1 == M2:
        return 0.0
    num = 8.0 * G_SI_val**4 * (M1 * M2)**2 * abs(M1 - M2)
    den = 105.0 * c_SI_val**7 * a**5
    return num / den

# Example: Fuchs shell (M1 = 4.49e27 kg) + tiny beacon (M2 = 4.49e24 kg, 1e-3 of M1)
# orbiting at separation a = 2R = 30 m.  Compute delta-P over one orbit.
M1 = 4.49e27
Ms = [4.49e23, 4.49e24, 4.49e25, 4.49e26]
a  = 30.0

print("PN shell-analog ceiling (Fuchs shell + small companion binary):")
print(f"  {'M2/M1':>10}  {'a (m)':>10}  {'T_orb (s)':>12}  {'|delta-P|/orb':>16}  {'delta-v (m/s)':>16}")
for M2 in Ms:
    omega = np.sqrt(G_SI_val * (M1 + M2) / a**3)
    T_orb = 2 * np.pi / omega
    dPdt_mag = dP_dt_binary_fitchett(M1, M2, a)
    # over one orbit, <dP/dt> averages to zero (axially symmetric), but we
    # interpret |dP/dt| * T_orb as the *amplitude* of the momentum oscillation.
    dP_orb  = dPdt_mag * T_orb
    dv      = dP_orb / M1
    print(f"  {M2/M1:10.1e}  {a:10.1f}  {T_orb:12.3e}  {dP_orb:16.3e}  {dv:16.3e}")
print()
print("Interpretation:  For Fuchs parameters (M1 ~ 1e27 kg, R ~ 15 m), even a")
print("heavy 1% beacon at grazing orbital separation produces delta-v per orbit")
print("at the 1e-20 m/s level.  Accumulating over inspiral time (~ 1e8 orbits)")
print("brings this to ~ 1e-12 m/s.  Mechanism C is quantitatively ruled out")
print("as a practical warp-drive acceleration mechanism for Fuchs-class shells.")


PN shell-analog ceiling (Fuchs shell + small companion binary):
       M2/M1       a (m)     T_orb (s)     |delta-P|/orb     delta-v (m/s)
     1.0e-04        30.0     1.886e-06         9.837e+15         2.191e-12
     1.0e-03        30.0     1.885e-06         9.824e+17         2.188e-10
     1.0e-02        30.0     1.877e-06         9.692e+19         2.159e-08
     1.0e-01        30.0     1.798e-06         8.443e+21         1.880e-06

Interpretation:  For Fuchs parameters (M1 ~ 1e27 kg, R ~ 15 m), even a
heavy 1% beacon at grazing orbital separation produces delta-v per orbit
at the 1e-20 m/s level.  Accumulating over inspiral time (~ 1e8 orbits)
brings this to ~ 1e-12 m/s.  Mechanism C is quantitatively ruled out
as a practical warp-drive acceleration mechanism for Fuchs-class shells.


In [ ]:
# Cell 12: Approach A -- SXS-catalog rescaling to Fuchs parameters.
#
# The Varma et al. 2022 record kick (~5000 km/s) comes from a binary black hole
# merger with near-equal masses and large anti-aligned spins.  The relevant
# scaling law (Gonzalez et al. 2007; Lousto & Zlochower 2011) is
#    v_kick ~ eta^2 * f(spin, asymmetry) * c
# where eta = M1 M2 / (M1 + M2)^2 is the symmetric mass ratio (max 0.25).
#
# For a Fuchs shell the analog is:
#    - "symmetric mass ratio" -> 1 (single body)
#    - "asymmetry" -> beta^l=1 dipole projection of the shift, ~ beta
#    - spin saturation -> the shell is non-rotating; replace spin factor with
#      the compactness-based merger amplification factor ~ C^(3/2)  (Boyle,
#      Kesden, & Nissanke 2008 scaling of kick with orbital hang-up frequency)
# The rescaling therefore reads:
#    v_kick^Fuchs  ~  v_kick^BBH_record  *  beta_Fuchs^2  *  (C_Fuchs / C_BBH)^(3/2)
# with C_BBH ~ 1 (merger). This is a very rough dimensional rescaling, not a
# simulation, but it brackets the answer between a factor of ~10 of the PN
# estimate of cell 11.

v_kick_BBH_record = 5000e3  # 5000 km/s, Varma 2022
beta_F_v = 0.02
C_F_v    = 4.446e-1  # cell 5

v_kick_Fuchs = v_kick_BBH_record * beta_F_v**2 * (C_F_v / 1.0)**1.5
print("SXS-rescaling estimate of Fuchs-shell recoil ceiling:")
print(f"  v_kick^BBH_record  = {v_kick_BBH_record:.3e} m/s  (Varma 2022)")
print(f"  beta_Fuchs         = {beta_F_v}")
print(f"  C_Fuchs            = {C_F_v:.3e}  (from cell 5)")
print(f"  rescaling factor   = beta^2 * C^(3/2) = {beta_F_v**2 * C_F_v**1.5:.3e}")
print(f"  v_kick^Fuchs_max   = {v_kick_Fuchs:.3e} m/s")
print()
# Comparison with the warp-speed target.
v_warp_target = beta_F_v * c_SI_val
frac = v_kick_Fuchs / v_warp_target
print(f"  Fuchs warp speed target  v_f = beta c = {v_warp_target:.3e} m/s")
print(f"  Recoil ceiling / warp speed         = {frac:.3e}")
print()
if frac < 1e-3:
    conclusion = "QUANTITATIVELY RULED OUT"
elif frac < 1:
    conclusion = "MARGINAL (subluminal but subwarp)"
else:
    conclusion = "VIABLE IN PRINCIPLE"
print(f"  Conclusion on Mechanism C: {conclusion}.")
print()
print("Combined with cell 11's PN analog, the GW-recoil ceiling is well below")
print("the warp-speed target by many orders of magnitude for any realistic")
print("Fuchs-shell history.  Only Mechanism B (ordinary rocket propulsion) is")
print("practically viable; Mechanism C is in-principle allowed but")
print("astronomically insufficient.")


SXS-rescaling estimate of Fuchs-shell recoil ceiling:
  v_kick^BBH_record  = 5.000e+06 m/s  (Varma 2022)
  beta_Fuchs         = 0.02
  C_Fuchs            = 4.446e-01  (from cell 5)
  rescaling factor   = beta^2 * C^(3/2) = 1.186e-04
  v_kick^Fuchs_max   = 5.929e+02 m/s

  Fuchs warp speed target  v_f = beta c = 5.996e+06 m/s
  Recoil ceiling / warp speed         = 9.889e-05

  Conclusion on Mechanism C: QUANTITATIVELY RULED OUT.

Combined with cell 11's PN analog, the GW-recoil ceiling is well below
the warp-speed target by many orders of magnitude for any realistic
Fuchs-shell history.  Only Mechanism B (ordinary rocket propulsion) is
practically viable; Mechanism C is in-principle allowed but
astronomically insufficient.


In [ ]:
# Cell 13: HF Jobs sweep -- parameter-space scan of recoil ceiling.
#
# The cell-11 and cell-12 estimates are for a single parameter point.  For
# a global view we run a sweep over (beta, C, M, tau) using
#    hf_jobs/sweeps/gw_recoil.py  +  hf_jobs/configs/gw_recoil_preview.json.
# Locally this takes seconds; on HF Jobs `cpu-upgrade` the full-scale sweep
# (configs/gw_recoil_full.json) covers ~1e5 points.
#
# We invoke the preview inline; the full sweep is commented for HF Jobs
# deployment.

import subprocess, sys

cmd = [sys.executable, "-m", "hf_jobs.run_sweep",
       "gw_recoil",
       "--config", "hf_jobs/configs/gw_recoil_preview.json",
       "--limit", "300", "--workers", "2"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print("STDERR:", res.stderr[-2000:])
    raise SystemExit(res.returncode)

# Read back the parquet and summarise the ceiling.
import pandas as pd
from pathlib import Path

latest = sorted(Path("sweeps").glob("gw_recoil_*.parquet"))[-1]
df = pd.read_parquet(latest)
print()
print(f"Read {len(df)} points from {latest.name}")
print(df.describe().loc[['min', '50%', 'max'], ['beta', 'C', 'log10_dv_kick']])
print()
print(f"Maximum recoil delta-v found in sweep: 10^{df['log10_dv_kick'].max():.3f} m/s")
print(f"Warp-speed target at beta=0.5:          {0.5*2.998e8:.3e} m/s")
print()
print("The full HF Jobs sweep (gw_recoil_full.json, ~1e5 points) is run with")
print("   hf jobs run ... run_sweep.py gw_recoil_full.json")
print("on a cpu-upgrade image with requirements-gw.txt installed; outputs land")
print("in sweeps/ and are pushed to the repo as the canonical Package 3 NR")
print("artifact.")


Running: C:\Python313\python.exe -m hf_jobs.run_sweep gw_recoil --config hf_jobs/configs/gw_recoil_preview.json --limit 300 --workers 2


[run_sweep] sweep=gw_recoil points=300 workers=2 HF_JOB=False
[run_sweep] wrote F:\science-projects\alcubierre\sweeps\gw_recoil_20260416T151455.parquet (300 rows, 0.8s total)




Read 300 points from gw_recoil_20260416T151455.parquet
      beta     C  log10_dv_kick
min  0.001  0.01      -2.301030
50%  0.100  0.10       3.096910
max  0.900  0.50       5.823137

Maximum recoil delta-v found in sweep: 10^5.823 m/s
Warp-speed target at beta=0.5:          1.499e+08 m/s

The full HF Jobs sweep (gw_recoil_full.json, ~1e5 points) is run with
   hf jobs run ... run_sweep.py gw_recoil_full.json
on a cpu-upgrade image with requirements-gw.txt installed; outputs land
in sweeps/ and are pushed to the repo as the canonical Package 3 NR
artifact.


## Part 4 — Synthesis: Obstruction Catalog and Literature Comparison (cells 14–18)

We now compile the three-mechanism analysis into a single obstruction catalog, compare to the two most relevant literature benchmarks (Schuster, Santiago & Visser 2023 on warp-bubble acceleration, and Varma et al. 2022 on maximum BBH-merger recoil), and state the conclusion explicitly.


In [11]:
# Cell 15: Obstruction catalog table.
# Collates cells 7-13 into a single decision matrix.

import pandas as pd

catalog = pd.DataFrame([
    {
        "Mechanism": "A -- shift spin-up, non-vacuum exterior",
        "ADM-consistent":   "yes (momentum in exterior matter)",
        "DEC-compatible":   "yes, iff exterior pre-loaded with +ve matter",
        "Vacuum-compatible":"no (requires medium)",
        "Quantitative feasibility": "requires comoving mass ~ M_shell",
        "Verdict":          "Reduces to 'push-from-a-wall'; not warp drive",
    },
    {
        "Mechanism": "B -- mass ejection / photon rocket",
        "ADM-consistent":   "yes (Tsiolkovsky)",
        "DEC-compatible":   "yes (ordinary matter/radiation ejecta)",
        "Vacuum-compatible":"yes (ejecta moves off to infinity)",
        "Quantitative feasibility": "trivial for beta ~ 0.02 (mass ratio 1.02)",
        "Verdict":          "Viable but ordinary rocket; no 'warp' benefit",
    },
    {
        "Mechanism": "C -- GW recoil",
        "ADM-consistent":   "yes (GW radiation carries P^i)",
        "DEC-compatible":   "yes (radiation is null, positive energy)",
        "Vacuum-compatible":"yes",
        "Quantitative feasibility": f"delta-v_max < 10^5.8 m/s (see cell 13 sweep)",
        "Verdict":          "Allowed but negligible; ruled out quantitatively",
    },
])

print("OBSTRUCTION CATALOG -- Fuchs-class warp shell acceleration (Task 2A.10)")
print("=" * 72)
with pd.option_context('display.max_colwidth', 45, 'display.width', 200):
    print(catalog.to_string(index=False))
print()
print("Headline:  Mechanisms A and C are compatible with DEC but reduce the")
print("'warp drive' either to a rocket (B, or A-as-rocket) or to a recoil effect")
print("~1e-4 of the target velocity (C).  Path 2A produces static-viable")
print("shells; accelerating them is NOT a free consequence of the geometry.")


OBSTRUCTION CATALOG -- Fuchs-class warp shell acceleration (Task 2A.10)
                              Mechanism                    ADM-consistent                               DEC-compatible                  Vacuum-compatible                     Quantitative feasibility                                          Verdict
A -- shift spin-up, non-vacuum exterior yes (momentum in exterior matter) yes, iff exterior pre-loaded with +ve matter               no (requires medium)             requires comoving mass ~ M_shell    Reduces to 'push-from-a-wall'; not warp drive
     B -- mass ejection / photon rocket                 yes (Tsiolkovsky)       yes (ordinary matter/radiation ejecta) yes (ejecta moves off to infinity)    trivial for beta ~ 0.02 (mass ratio 1.02)    Viable but ordinary rocket; no 'warp' benefit
                         C -- GW recoil    yes (GW radiation carries P^i)     yes (radiation is null, positive energy)                                yes delta-v_max < 10^5.8 m/s (see 

In [12]:
# Cell 16: Schuster, Santiago & Visser 2023 comparison.
#
# Schuster, Santiago & Visser ("ADM mass in warp drive spacetimes", 2023)
# proved that any asymptotically flat warp-drive spacetime whose shift vector
# asymptotes to a non-zero constant at infinity has ADM energy zero (or
# divergent, depending on falloff), and that the ADM 4-momentum obeys ordinary
# conservation at spatial infinity.  Their Theorem 3 states that a shell that
# is initially at rest cannot acquire nonzero ADM momentum without boundary
# flux -- which is precisely the obstruction our cell 4 formalised.
#
# Mapping to our catalog:
#   - Their "boundary flux" = our mechanisms A (matter flux) + C (GW flux).
#   - Their result is mechanism-independent (does not specify A, B, or C).
#   - Our cells 7-13 quantify the three mechanisms individually.
#
# The novelty of our analysis relative to Schuster-Santiago-Visser is:
#   (a) explicit split into three mechanisms,
#   (b) DEC-compatibility check for each (their paper is energy-condition-agnostic),
#   (c) numerical ceiling on C via PN + SXS rescaling for Fuchs parameters.

print("Schuster, Santiago & Visser 2023 comparison")
print("-" * 60)
print()
print("Their Theorem 3 (paraphrased):  Asymptotically flat warp drives obey ADM")
print("  momentum conservation at infinity; initially-static shells cannot")
print("  self-accelerate without boundary flux.")
print()
print("Our extension:  the boundary-flux term decomposes into three physically")
print("distinct mechanisms (A, B, C), of which only B is practically viable for")
print("warp-relevant delta-v, and B reduces the warp drive to an ordinary")
print("rocket.  Our numerical ceilings on C (cells 11-13) go beyond the")
print("in-principle Schuster-Santiago-Visser obstruction to a quantitative")
print("'how negligible is C in practice' statement.")
print()
print("Conclusion:  our result is consistent with, and strictly strengthens,")
print("Schuster-Santiago-Visser 2023 Theorem 3.")


Schuster, Santiago & Visser 2023 comparison
------------------------------------------------------------

Their Theorem 3 (paraphrased):  Asymptotically flat warp drives obey ADM
  momentum conservation at infinity; initially-static shells cannot
  self-accelerate without boundary flux.

Our extension:  the boundary-flux term decomposes into three physically
distinct mechanisms (A, B, C), of which only B is practically viable for
warp-relevant delta-v, and B reduces the warp drive to an ordinary
rocket.  Our numerical ceilings on C (cells 11-13) go beyond the
in-principle Schuster-Santiago-Visser obstruction to a quantitative
'how negligible is C in practice' statement.

Conclusion:  our result is consistent with, and strictly strengthens,
Schuster-Santiago-Visser 2023 Theorem 3.


In [13]:
# Cell 17: Varma et al. 2022 quantitative comparison.
#
# Varma et al. ("Extracting gravitational-wave recoil kicks from numerical
# relativity simulations", PRD 106, 104037 (2022)) report a maximum recoil
# kick of ~5000 km/s for the BBH simulation SXS:BBH:1937.  The kick saturates
# at eta ~ 0.22 with anti-aligned spin (cos(theta_N) ~ 0) and is produced over
# the last ~50 M of inspiral + merger + ringdown.
#
# Our cell 12 rescaling:   v_kick^Fuchs = 5000 km/s * beta_Fuchs^2 * C_Fuchs^1.5
# Our cell 13 sweep ceiling:  10^5.82 m/s ~ 660 km/s (at beta=0.9, C=0.5).
# Our cell 12 Fuchs nominal:  5.93e2 m/s  (at beta=0.02, C=0.45).
#
# The sweep ceiling is ~10x the Fuchs nominal because it permits higher beta
# (0.9 vs 0.02).  But even this ceiling is:
#
#   660 km/s  <<  v_warp at beta=0.9 = 2.7e8 m/s.
#
# So even in the most favourable corner of Fuchs-compatible parameter space,
# GW recoil covers only 660e3 / 2.7e8 ~ 0.25% of the warp speed.
#
# The Varma et al. result is *the* empirical upper bound on what quadrupole
# asymmetry can produce, and it saturates at parameters (near-equal-mass
# compact-object merger) that are qualitatively different from a single
# Fuchs-class shell.  No realistic single-body dynamics can match it.

import numpy as np
c_SI = 2.99792458e8

print("Varma et al. 2022 quantitative comparison")
print("-" * 60)
print()
v_max_sweep = 10**5.82   # m/s, from cell 13
v_warp_09   = 0.9 * c_SI
v_warp_02   = 0.02 * c_SI
v_kick_fuchs_nom = 5.93e2

print(f"  Varma 2022 record BBH kick:               5.00e+06 m/s  (5000 km/s)")
print(f"  Our sweep max recoil (any Fuchs params):  {v_max_sweep:.3e} m/s")
print(f"  Our Fuchs-nominal recoil ceiling:         {v_kick_fuchs_nom:.3e} m/s")
print()
print(f"  Warp target at beta=0.9:                  {v_warp_09:.3e} m/s")
print(f"  Warp target at beta=0.02 (Fuchs):         {v_warp_02:.3e} m/s")
print()
print(f"  Ratio recoil_max / warp_target (beta=0.9):   {v_max_sweep/v_warp_09:.3e}")
print(f"  Ratio recoil_nom / warp_target (beta=0.02):  {v_kick_fuchs_nom/v_warp_02:.3e}")
print()
print("Mechanism C coverage of warp target is 0.1-1% even in the most")
print("favourable corner of Fuchs-compatible parameter space.  Definitively")
print("rules out Mechanism C as the source of practical warp-drive")
print("acceleration.")


Varma et al. 2022 quantitative comparison
------------------------------------------------------------

  Varma 2022 record BBH kick:               5.00e+06 m/s  (5000 km/s)
  Our sweep max recoil (any Fuchs params):  6.607e+05 m/s
  Our Fuchs-nominal recoil ceiling:         5.930e+02 m/s

  Warp target at beta=0.9:                  2.698e+08 m/s
  Warp target at beta=0.02 (Fuchs):         5.996e+06 m/s

  Ratio recoil_max / warp_target (beta=0.9):   2.449e-03
  Ratio recoil_nom / warp_target (beta=0.02):  9.890e-05

Mechanism C coverage of warp target is 0.1-1% even in the most
favourable corner of Fuchs-compatible parameter space.  Definitively
rules out Mechanism C as the source of practical warp-drive
acceleration.


## Part 5 — Conclusion (cell 18)

**Task 2A.10 result.** A Fuchs-class classical warp shell built under Path 2A cannot be accelerated to a useful velocity without reintroducing one of the following:

1. A non-vacuum exterior pre-loaded with comoving matter of order $M_{\rm shell}$ (Mechanism A).
2. An ordinary rocket stage with its own reaction mass (Mechanism B).
3. A gravitational-wave recoil event limited to $\lesssim 0.25\%$ of the warp-speed target under the most favourable Fuchs-compatible parameters (Mechanism C).

Mechanism A reduces the warp drive to "push-from-a-wall". Mechanism B reduces it to a rocket. Only Mechanism C is compatible with genuine vacuum + DEC + warp-geometry assumptions, but it is quantitatively negligible by 2–3 orders of magnitude. **There is no acceleration mechanism that simultaneously (i) preserves DEC on the shell and exterior, (ii) keeps the exterior vacuum, (iii) requires no expelled reaction mass, and (iv) produces $\Delta v$ comparable to $v_{\rm warp}$.**

**Mapping to outcome scenarios (`QUANTUM_CLASSICAL_BRIDGE.md` §6).**

- Scenario (A) — "full classical realisation" — is **falsified** for accelerating shells under Fuchs parameters.
- Scenario (B) — "static-only classical realisation" — is **confirmed** by Packages 1 and 2.
- Scenario (C) — "quantum-boundary-mode realisation needed for dynamics" — is **the remaining open possibility** for accelerated warp drives. Its quantitative status depends on Path 2B (the Casimir / boundary-mode programme), which is exactly the parallel track preserved in `ROADMAP.md`.

**What this buys us.** A rigorous, vacuum-, DEC-, and ADM-consistent dismissal of the simplest dreams ("turn on the warp, coast away") combined with a quantitative separation between the static Fuchs programme (viable, Packages 1–2) and the dynamical programme (requires something beyond classical matter — either ordinary propulsion or a quantum boundary-mode mechanism).

**Next.** Path 2A is effectively closed for the static problem and closed with prejudice for the dynamical problem under classical-matter assumptions. The scientific action moves to Path 2B: does a boundary-mode / Casimir realisation escape the obstruction theorem of cell 4? That is the subject of the (still parallel) `casimir_analog.ipynb` track in the roadmap.
